In [ ]:
import torch
import torchvision
from torchvision import transforms as torchtrans 
import cv2
import os 
import pandas as pd 
import numpy as np
# defining the files directory and testing directory
train_dir = "../../amia-public-challenge-2026/train/train"
test_dir = "../../amia-public-challenge-2026/test/test"
image_size_df= pd.read_csv("../../amia-public-challenge-2026/img_size.csv")

train_csv_path= "../../amia-public-challenge-2026/train.csv"
RANDOM_SEED= 42
df= pd.read_csv(train_csv_path)


from sklearn.model_selection import train_test_split

image_ids = df["image_id"].unique()

train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=RANDOM_SEED
)


class ChestX_RAYDataset(torch.utils.data.Dataset):

    def __init__(self, files_dir, image_ids, transforms=None):
        self.transforms = transforms
        self.files_dir = files_dir

        # sorting the images for consistency
        # To get images, the extension of the filename is checked to be png
        self.imgs = image_ids#[image for image in os.listdir(self.files_dir)if image[-4:]=='.png']
        self.df = df
        self.image_size_df = image_size_df

    def __getitem__(self, idx):

        img_name = self.imgs[idx]
        #image_path = os.path.join(self.files_dir, img_name)

        # reading the images and converting them to correct color and format
        image_path = os.path.join(self.files_dir,img_name + ".png")
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        img_res= (torch.from_numpy(img_rgb).permute(2,0,1).float())/255.0  #scaling, permute rearranges dimensions cz opencv reads in HWC format but pytorch expects CHW format 
        
        boxes = []
        labels = []
        
        # box coordinates for files are extracted and corrected for image size given
        image_size= self.image_size_df.loc[self.image_size_df['image_id'] == img_name]
        x, y= image_size['dim0'], image_size['dim1']
        original_x, original_y= x.values[0], y.values[0]
        x_scale= 1024/original_x
        y_scale= 1024/original_y
        for instance in self.df.loc[self.df['image_id'] == img_name].itertuples():
            if instance.class_id == 14: 
                continue
            labels.append(instance.class_id + 1) #Faster R-CNN reserves 0 = background
            # bounding box
            xmin = instance.x_min
            xmax = instance.x_max
            
            ymin = instance.y_min
            ymax = instance.y_max

            xmin_corr = xmin*x_scale
            xmax_corr = xmax*x_scale
            ymin_corr = ymin*y_scale
            ymax_corr = ymax*y_scale
            
            boxes.append([xmin_corr, ymin_corr, xmax_corr, ymax_corr])
        
        # convert boxes into a torch.Tensor
        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # getting the areas of the boxes
        #area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        # suppose all instances are not crowd
        #iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)


        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        #target["area"] = area
        #target["iscrowd"] = iscrowd
        # image_id
        image_id = torch.tensor([idx])
        target["image_id"] = image_id
        target["image_name"] = img_name


        if self.transforms:
            
            sample = self.transforms(image = img_res,
                                     bboxes = target['boxes'],
                                     labels = labels)
            
            img_res = sample['image']
            target['boxes'] = torch.Tensor(sample['bboxes'])
            
            
            
        return img_res, target

    def __len__(self):
        return len(self.imgs)


# check dataset
train_dataset = ChestX_RAYDataset(train_dir, train_ids)

val_dataset = ChestX_RAYDataset(train_dir, val_ids)
print('length of training dataset = ', len(train_dataset), '\n')
print('length of validation dataset = ', len(val_dataset), '\n')

# getting the image and target for a test index.  Feel free to change the index.
img, target = train_dataset[78]
val_img, val_target = val_dataset[9]
print('train dataset')
print(img.shape)
print(target["boxes"].shape)
print(target["labels"])
print("val dataset")
print(img.shape)
print(target["boxes"].shape)
print(target["labels"])

length of training dataset =  6858 

length of validation dataset =  1715 

train dataset
torch.Size([3, 1024, 1024])
torch.Size([0, 4])
tensor([], dtype=torch.int64)
val dataset
torch.Size([3, 1024, 1024])
torch.Size([0, 4])
tensor([], dtype=torch.int64)


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

In [7]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
NUM_CLASSES = 15 #14 classes + background
# load a model pre-trained pre-trained on COCO
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
# get number of input features for the classifier
in_features = model.roi_heads.box_predictor.cls_score.in_features
# replace the pre-trained head with a new one
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES) 

In [8]:
# to train on gpu if selected.
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')


# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.01,
                            momentum=0.9, weight_decay=0.0005)

# and a learning rate scheduler which decreases the learning rate by
# 10x every 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                               step_size=3,
                                               gamma=0.1)

In [9]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm

num_epochs = 5

for epoch in tqdm(range(num_epochs)):
    epoch_loss = 0.0

    # training
    model.train()

    for images, targets in tqdm(train_loader):

        images = [img.to(device) for img in images]

        targets = [
            {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in t.items()
            }
            for t in targets
        ]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        epoch_loss += losses.item()
        print(f"Train Loss: {epoch_loss/len(train_loader):.4f}")

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    lr_scheduler.step()

    # validation
    model.eval()

    metric = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.4, 0.5, 0.75])

    with torch.no_grad():

        for images, targets in tqdm(val_loader):

            images = [img.to(device) for img in images]

            targets = [
                {
                    k: v.to(device) if isinstance(v, torch.Tensor) else v
                    for k, v in t.items()
                }
                for t in targets
            ]

            predictions = model(images)

            preds_cpu = [
                {k: v.cpu() for k, v in p.items()}
                for p in predictions
            ]

            targets_cpu = [
                {
                    k: v.cpu() if isinstance(v, torch.Tensor) else v
                    for k, v in t.items()
                }
                for t in targets
            ]

            metric.update(preds_cpu, targets_cpu)

    results = metric.compute()

    print(f"\nEpoch {epoch+1}")
    print(f"mAP:    {results['map']:.4f}")
    print(f"mAP50:  {results['map_50']:.4f}")
    print(f"mAP75:  {results['map_75']:.4f}")
    


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/1715 [00:00<?, ?it/s]

Train Loss: 0.0019
Train Loss: 0.0026
Train Loss: 0.0031
Train Loss: 0.0034
Train Loss: 0.0039
Train Loss: 0.0049
Train Loss: 0.0050
Train Loss: 0.0053
Train Loss: 0.0053
Train Loss: 0.0060
Train Loss: 0.0062
Train Loss: 0.0064
Train Loss: 0.0065
Train Loss: 0.0066
Train Loss: 0.0069
Train Loss: 0.0072
Train Loss: 0.0073
Train Loss: 0.0076
Train Loss: 0.0079
Train Loss: 0.0081
Train Loss: 0.0082
Train Loss: 0.0084
Train Loss: 0.0085
Train Loss: 0.0085
Train Loss: 0.0091
Train Loss: 0.0093
Train Loss: 0.0094
Train Loss: 0.0095
Train Loss: 0.0104
Train Loss: 0.0105
Train Loss: 0.0107
Train Loss: 0.0109
Train Loss: 0.0110
Train Loss: 0.0115
Train Loss: 0.0119
Train Loss: 0.0122
Train Loss: 0.0124
Train Loss: 0.0126
Train Loss: 0.0127
Train Loss: 0.0129
Train Loss: 0.0132
Train Loss: 0.0136
Train Loss: 0.0137
Train Loss: 0.0146
Train Loss: 0.0147
Train Loss: 0.0151
Train Loss: 0.0152
Train Loss: 0.0153
Train Loss: 0.0154
Train Loss: 0.0156
Train Loss: 0.0156
Train Loss: 0.0159
Train Loss: 

  0%|          | 0/429 [00:00<?, ?it/s]


Epoch 1
mAP:    0.0320
mAP50:  0.0372
mAP75:  0.0027


...


Epoch 4
mAP:    0.0753
mAP50:  0.0887
mAP75:  0.0118


  0%|          | 0/1715 [00:00<?, ?it/s]

Train Loss: 0.0002
Train Loss: 0.0004
Train Loss: 0.0007
Train Loss: 0.0008
Train Loss: 0.0010
Train Loss: 0.0011
Train Loss: 0.0012
Train Loss: 0.0015
Train Loss: 0.0016
Train Loss: 0.0019
Train Loss: 0.0019
Train Loss: 0.0021
Train Loss: 0.0022
Train Loss: 0.0023
Train Loss: 0.0026
Train Loss: 0.0028
Train Loss: 0.0028
Train Loss: 0.0030
Train Loss: 0.0030
Train Loss: 0.0031
Train Loss: 0.0037
Train Loss: 0.0040
Train Loss: 0.0041
Train Loss: 0.0044
Train Loss: 0.0044
Train Loss: 0.0045
Train Loss: 0.0048
Train Loss: 0.0053
Train Loss: 0.0053
Train Loss: 0.0053
Train Loss: 0.0058
Train Loss: 0.0059
Train Loss: 0.0063
Train Loss: 0.0069
Train Loss: 0.0072
Train Loss: 0.0075
Train Loss: 0.0078
Train Loss: 0.0080
Train Loss: 0.0081
Train Loss: 0.0084
Train Loss: 0.0086
Train Loss: 0.0092
Train Loss: 0.0093
Train Loss: 0.0096
Train Loss: 0.0099
Train Loss: 0.0099
Train Loss: 0.0100
Train Loss: 0.0100
Train Loss: 0.0103
Train Loss: 0.0103
Train Loss: 0.0105
Train Loss: 0.0106
Train Loss: 

  0%|          | 0/429 [00:00<?, ?it/s]


Epoch 5
mAP:    0.0793
mAP50:  0.0902
mAP75:  0.0114


In [10]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

# -------------------------
# CONFIG
# -------------------------

TEST_DIR = "../../amia-public-challenge-2026/test/test"
CHECKPOINT_PATH = "model.pth"
CSV_OUTPUT = "submission.csv"

# confidence threshold
SCORE_THRESHOLD = 0.3

# class id for "No finding"
NO_FINDING_CLASS_ID = 14

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LOAD MODEL
# -------------------------

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.to(device)
model.eval()

# -------------------------
# TRANSFORMS
# -------------------------

transform = transforms.ToTensor()

# -------------------------
# INFERENCE
# -------------------------

results = []

test_images = sorted(os.listdir(TEST_DIR))

with torch.no_grad():

    for image_name in tqdm(test_images):

        image_path = os.path.join(TEST_DIR, image_name)

        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).to(device)

        prediction = model([image_tensor])[0]

        boxes = prediction["boxes"].cpu()
        scores = prediction["scores"].cpu()
        labels = prediction["labels"].cpu()

        prediction_strings = []

        for box, score, label in zip(boxes, scores, labels):

            score = float(score)

            # filter low-confidence predictions
            if score < SCORE_THRESHOLD:
                continue

            xmin, ymin, xmax, ymax = box.tolist()

            label = int(label)

            pred_str = (
                f"{label} "
                f"{score:.4f} "
                f"{xmin:.1f} {ymin:.1f} "
                f"{xmax:.1f} {ymax:.1f}"
            )

            prediction_strings.append(pred_str)

        # if NO detections remain
        if len(prediction_strings) == 0:

            prediction_string = (
                f"{NO_FINDING_CLASS_ID} 1.0 0 0 1 1"
            )

        else:

            prediction_string = " ".join(prediction_strings)

        # remove extension from image id
        image_id = os.path.splitext(image_name)[0]

        results.append({
            "image_id": image_id,
            "PredictionString": prediction_string
        })

# -------------------------
# SAVE CSV
# -------------------------

submission_df = pd.DataFrame(results)

submission_df.to_csv(CSV_OUTPUT, index=False)

print(submission_df.head())
print(f"\nSaved to {CSV_OUTPUT}")

  0%|          | 0/6427 [00:00<?, ?it/s]

                           image_id  \
0  00X4Pb5TcOhWWwrDwn9UoRDJhwYRuusp   
1  00eCz0yTwisqK7dgZKrdhLh4cMP9FewR   
2  00wsXaGGLhOo977BBHmhbKVNu02fWdPl   
3  02IEFam0BlSztSMY3YeA9svnDJOxTKDg   
4  02fQeJYiEhOeebwkwE8wsD0FPyz8EWHD   

                                    PredictionString  
0  1 0.5919 517.7 281.2 644.7 423.8 4 0.4753 355....  
1                                     14 1.0 0 0 1 1  
2                                     14 1.0 0 0 1 1  
3  1 0.8415 554.0 264.0 644.3 354.3 6 0.4749 602....  
4                                     14 1.0 0 0 1 1  

Saved to submission.csv
